In [ ]:

import math
import numpy as np
import random
from numpy.random import uniform, exponential
import seaborn as sns
import time,timeit
from typing import Any, Callable, List, Optional, Tuple


def simulate(X,n_sim,ddof=0,**kwargs):
    """
    Parameters:
    -----------
    X : callable
        A function (or callable object) that generates a single simulated value when called with **kwargs.
    n_sim : int
        Number of simulations to run.
    ddof : int, optional (default=0)
        Delta degrees of freedom for variance and standard deviation calculations.
        If ddof=0 (default), computes population variance/std.
        If ddof=1, computes sample variance/std (unbiased estimator).
    **kwargs : dict
        Additional keyword arguments passed to the function X.

    Returns:
    --------
    s : ndarray
        Array of all simulated values (shape: (n_sim,)).
    mean_sim : float
        Mean of the simulated values.
    median_sim : float
        Median of the simulated values.
    var_sim : float
        Variance of the simulated values (with specified ddof).
    std_sim : float
        Standard deviation of the simulated values (with specified ddof).
    """
    s = np.empty(n_sim)
    for i in range(n_sim):
      s[i]=X(**kwargs)
    mean_sim = np.mean(s)
    median_sim = np.median(s)
    var_sim = np.var(s,ddof=ddof)
    std_sim = np.std(s,ddof=ddof)
    return s,mean_sim,median_sim,var_sim,std_sim

def cumulativa(s,k):
  return np.sum(s <= k)/len(s)


"""

El metodo de aceptación y rechazo consiste en generar una variable aleatoria X a partir de otra variable aleatoria Y que sabemos generar
y que tiene una funcion de densidad g(x) conocida.
Esta función g(x) debe cumplir la siguiente condición: f(x) <= c*g(x) donde f(x) != 0 y con c una constante mayor que 1.
Equivalentemente, se puede despejar la constante c y obtener la expresión f(x)/g(x) <= c
La constante c deberia ser elegida como algún maximo local de ese cociente dentro del intervalo donde queremos generar la variable aleatoria.

simy: función que me genera la Y conocida contra la que rechazo la VA que quiero genererar.
Px: densidad de X
Py: densidad de Y
c: constante c del cociente, si no la paso defaultea a 1 que es el mínimo.
valores_x: valores que asume la variable X. Esto se pasa en caso de que los valores de x estén definidos explícitamente y sean finitos.

"""


def rechazo(simy, px, py, c: Optional[float] = None, valores_x: Optional[List[float | int]] = None):
  #Si no le paso el c, lo tengo que calcular como el maximo del cociente, y debería ser la menor cota posible para reducir la cantidad de rechazos al mínimo.
  if c is None:
    assert valores_x is not None and len(valores_x) != 0
    c = max(px(x) / py(x) for x in valores_x) #Máximo local alcanzable por el cociente para todos los valores donde está definida.
    c = max(c, 1) #Tiene que ser al menos 1, porque si fuese igual entonces X e Y tendrían la misma distribución y sabríamos generar X igual que Y.

  #Salgo de este while solo cuando "y" asume un valor posible de X, porque sino el cociente da cero por ser Px(y) = 0, y U no puede ser menor que cero.
  while True:
    y = simy() #Esto toma los valores que puede tomar X, y algunos mas que no importan.
    U = random.random()
    #Px(y) es la probabilidad de que X tome el valor que acabo de generar, y Py(y) lo mismo pero con Y, ambas positivas.
    if U < px(y) / (c * py(y)):
      return y
    #La probabilidad de aceptar un valor cuando "y" asume el valor 1/3 es

#Simula una poisson usando la exponencial.
def Poisson_con_exp(lamda):
 X = 0
 Producto = 1 - random.random()
 cota = math.exp(-lamda)
 while Producto >= cota:
  Producto *= 1 - random.random()
  X += 1
 return X

#Simula una gamma de parametros n y media 1/lamda
def Gamma(n,lamda):
  U = 1
  for _ in range(n):
    U *= 1-random.random()
  return -math.log(U)/lamda

def exponencial(lamda):
  return -math.log(random.random())/lamda



def eventosPoisson(lamda,T):
  """
  Devuelve len(eventos) que es el número de eventos que ocurren
  hasta el tiempo T. El i-ésimo elemento de Eventos(Eventos[i−1]) indica el tiempo en que
  ocurre elevento i, 1≤i≤NT
  """
  t=0
  Eventos=[]

  while t<T:
    t+=exponencial(lamda)
    if t<=T:
      Eventos.append(t)
  return len(Eventos), Eventos


def eventosPoissonNoHomogeneo(lamda, lamda_t, T): #Toma la media, la función de intensidad y el tiempo hasta el cual cuento.7
    eventos = []
    U = 1 - random()
    t = -math.log(U) / lamda
    while t < T:
        v = random()
        if v <= lamda_t(t)/lamda:
            eventos.append(t)
        U = 1 - random()
        t += -math.log(U) / lamda
    return len(eventos), eventos

def Poisson_no_homogeneo_adelgazamiento(T, lamda, lamda_t):
 'Devuelve el número de eventos y los tiempos en Eventos'
 'lamda_t(t): intensidad, lamda_t(t)<=lamda'

 Eventos = []
 U = 1-random()
 t =-math.log(U) / lamda
 while t <= T:
  V = random()
  if V < lamda_t(t) / lamda:
    Eventos.append(t)
  t += -math.log(1-random()) / lamda
 return len(Eventos), Eventos


"""
Intervals tiene que ser de la forma [(cota, lambda)] donde la cota es el valor maximo de la funcion en el intervalo y lambda hasta donde cuento los eventos con esa cota.
La idea es ir variando la cota según el intervalo de la funcion de densidad para evitar tener muchos rechazos en funciones que tienen minimos y maximos
muy abruptos y separados. Cada intervalo tiene su propia cota, que es el máximo local de la funcion de densidad.
"""
def poisson_no_homogeneo_adelgazamiento_mejorado(T, lamda_t, intervals):
    j = 0
    events = []
    t = exponential(lamda=intervals[j][0])
    while t <= T:
        lamda = intervals[j][0]
        b = intervals[j][1]

        if t <= b:
            v = random.random()
            if v < lamda_t(t)/lamda:
                events.append(t)
            t += exponential(lamda)

        else:  # t > b
            t = b + ((t-b) * lamda) / intervals[j+1][0]
            j += 1
    return len(events), events


#T tiempo hasta el cual cuento.
def Poisson_adelgazamiento_mejorado(T):
 interv = [2, 4, 6]
 lamda = [5, 9, 13]
 j = 0 #recorre subintervalos.
 t =-math.log ( 1- random() ) / lamda[j]
 NT = 0
 Eventos = []
 while t <= T:
  if t <= interv[j]:
    V = random.random()
    if V < (2 * t + 1) / lamda[j]:
      NT += 1
      Eventos.append(t)
      t +=-math.log(1- random()) / lamda[j]
  else: #t > interv[j]
    t = interv[j] + (t- interv[j]) * lamda[j] / lamda[j + 1]
    j += 1
 return NT, Eventos


def composicion_new(ps: List[float], sims: List[Callable[[], float]]):

    assert len(ps) == len(sims) != 0
    assert sum(ps) == 1
    acum = []
    u = random()
    i, acc = 0, ps[0]

    while u >= acc:
        i += 1
        acc += ps[i]

    return sims[i]()


#Otra forma idéntica al pseudocodigo.
def composicion_simple(probabilidades, variables): #Probabilidades de sortear cada variable, variables.

    acumulada = []
    U = random.random()
    i = 0
    acc = probabilidades[0] #Probabilidades de que se sortee cada variable.

    while U >= acc:
        i += 1
        acc += probabilidades[i]

    #Usa especificamente una exponencial con la variable sorteada, pero podría ser cualquier distribución.
    return exponencial(variables[i])


def normal_metodo_polar(mu, sigma):

  Rcuadrado = -2 * math.log( 1 - random() )
  Theta= 2 * math.pi * random()
  X= math.sqrt(Rcuadrado) * math.cos(Theta)
  Y= math.sqrt(Rcuadrado) * math.sin(Theta)
  return (X * sigma + mu, Y * sigma + mu)

def normal_ross(mu, sigma):
    # Primero genero |Z|

    def generarZ():

        while True:
            u = random()
            y = -math.log(random())

            if u <= math.exp(-((y-1)**2) / 2):
                return y

    generarMenosZ = lambda : -generarZ()

    # Después usamos composición para obtener Z a partir de componer |Z| y -|Z|
    return composicion_new([1/2, 1/2], [generarZ, generarMenosZ]) * sigma + mu

def generar_normal_con_exponenciales():
  while True:
    Y1 = -math.log(random.random())
    Y2 = -math.log(random.random())
    if Y2 >= (Y1-1)**2/2:
      if u < 0.5:
        return Y1
      return -Y1



def ejercicio1():

  c = 2.835

  #Densidad de X
  def px(x):
    return 30 * ((x**2) - (2 * x**3) + (x**4))

  #Variable contra la que voy a rechazar, una poisson de media 1
  def sim_y(lamda):
    X = 0
    Producto = 1 - random.random()
    cota = math.exp(-lamda)
    while Producto >= cota:
      Producto *= 1 - random.random()
      X += 1
    return X

  #Densidad de Y, una poisson de media 1.
  def py(x,lamda):
    return ((math.e**(-lamda))*(lamda**x))/(math.factorial(x))

  #Loop donde decido si acepto o rechazo el valor generado por Y según si es un posible valor de X o no.
  while True:
    y = sim_y(1)
    U = random.random()

    if U < px(y) / (c * py(y,1)):
      return y

resultado = lambda: ejercicio1()
start = time.time()
s, esperanza, _, varianza, desvio = simulate(resultado,n_sim=10000)

print("--------------TEST--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)


--------------TEST--------------
Tiempo:  0.058972835540771484
Esperanza:  2.3922
Varianza:  0.45197915999999994
Desvio:  0.6722939535649566


Para generar una variable aleatoria mediante el método de la transformada inversa primero buscamos la funcion de distribución acumulada y calculamos su inversa. El resultado retornado va a ser la inversa evaluada en un valor random.

Ejercicio 1


In [ ]:
"""
Calculo la integral desde el límite inferior del intervalo hasta x de s (calculo F(X)), y evalúo el resultado en el límite superior del intervalo
para saber en que intervalos se mueve U, porque la uso para evaluar la inversa de esa integral que calculé. Esto para cada intervalo de la f(x) original.
"""

def inciso_a():
  u = random.random()
  if u < 0.25:
    return 2 + 2*np.sqrt(u)
  else:
    return 6 - 2*np.sqrt(3*(1-u))


Ejercicio 2: Desarrolle métodos para generar las siguientes variables aleatorias

1.   Elemento de lista
2.   Elemento de lista



i) Distribución Pareto

ii) Distribución Erlang

iii) Distribución Weibull

In [ ]:
#i) Por transformada inversa.
#Primero busco su inversa.
def transformada_inversa_especial(a):
  i=1#Arranca en 1
  prob = 0

  U = random.random()

  while U>=prob:
    prob += (1-U) ** (1/(-a))
    i+=1
  return i

resultado = lambda: transformada_inversa_especial(2)
start = time.time()
s, esperanza, _, varianza, desvio = simulate(resultado,n_sim=10000)

print("--------------TEST--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)


--------------TEST--------------
Tiempo:  0.010420560836791992
Esperanza:  2.0
Varianza:  0.0
Desvio:  0.0


Ejercicio 3: Método de la composición



In [ ]:
#PREGUNTAR

#Pseudocódigo

"""
U = random.random() aleatoria.
alpha = [a1,a2,a3...an] lista de coeficientes.
i = 1
F = a1
while U >= F:
  i += 1
  F += alpha[i]
return Xi
"""

def exponencial(lamda):
    u = random.random()
    return ((-1) * math.log(1-u)) / lamda


variables = [3,5,7]
alpha = [0.5, 0.3, 0.2]



"Se sortea un Alpha y a ese Alpha le sorteamos Xi"

"""
 El método de composición permite generar una variable aleatoria X con función de probabilidad de masa dada por:
 P(X =j)=αpj + (1−α)qj con α en el intervalo [0,1] y j = 0,1,2,3,...
 Sacar el resto del apunte.
"""

#Otra forma idéntica al pseudocodigo.
def composicion_simple(probabilidades, variables): #Probabilidades, variables.

    acumulada = []
    U = random.random()
    i = 0
    acc = probabilidades[0]

    while U >= acc:
        i += 1
        acc += probabilidades[i]

    return exponencial(variables[i])

resultado2 = lambda: composicion_simple(alpha, variables)
start = time.time()
s, esperanza, _, varianza, desvio = simulate(resultado2,n_sim=10000)

print("--------------TEST--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)

--------------TEST--------------
Tiempo:  0.008930206298828125
Esperanza:  0.2553084008057029
Varianza:  0.08047721930589789
Desvio:  0.28368507064330667


EJERCICIO 4

In [ ]:

"""
La idea es que la densidad de esta función está definida como una integral, que es una sumatoria de infinitos términos.

Vemos que el término de la integral es x**y * exp(-y) y la variable cuantificada es y

Por lo que podemos ver esto como una composición de variables

X_y con función de probabilidad acumulada x**y

Se puede ver la función de probabilidad acumulada de X como:

F(x) = integral de 0 a inf de F_y(x) * e**(-y) dy

e**(-y) se puede ver como la densidad de una exponencial con parámetro 1, por lo que vemos que

integral de 0 a inf de e**(-y) dy = 1
"""

def ej4():

    y = exponential() #Esto me determina cual X elijo en la composición. Exponencial de media 1. Me determina e**y
    u = random.random()

    return u ** (1/y) #Como yo ahí sortee Y = e**y y quiero X, entonces devuelvo la raiz.

resultado2 = lambda: ej4()
start = time.time()
s, esperanza, _, varianza, desvio = simulate(resultado2,n_sim=10000)

print("--------------TEST--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)

--------------TEST--------------
Tiempo:  0.018233537673950195
Esperanza:  0.39965398739454233
Varianza:  0.11475889651915977
Desvio:  0.33876082494757237


Ejercicio 6:



In [ ]:
def sim_y():
  return random.random() #Uniforme contra la que rechazo.

def metodo1_maximos(n):
    return max(random.random() for _ in range(n))

def metodo2_inversa(n):
    return random.random() ** (1/n)

def metodo3_rechazo(n):
    return rechazo(sim_y, lambda x : n * x**(n-1), lambda x : 1, n) #Hay que pasarle la derivada porque nos dan F(x) y esto come f(x)


resultado1 = lambda: metodo1_maximos(5)
start = time.time()
s, esperanza, _, varianza, desvio = simulate(resultado1,n_sim=10000)

print("--------------TEST 1--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)

resultado2 = lambda: metodo2_inversa(5)
start = time.time()
s, esperanza, _, varianza, desvio = simulate(resultado2,n_sim=10000)

print("--------------TEST 2--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)

resultado3 = lambda: metodo3_rechazo(5)
start = time.time()
s, esperanza, _, varianza, desvio = simulate(resultado3,n_sim=10000)

print("--------------TEST 3--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)

--------------TEST 1--------------
Tiempo:  0.014354228973388672
Esperanza:  0.8339278648286514
Varianza:  0.0200215220577141
Desvio:  0.14149742774239432
--------------TEST 2--------------
Tiempo:  0.005641937255859375
Esperanza:  0.8322837173009408
Varianza:  0.019644361275251487
Desvio:  0.1401583435805785
--------------TEST 3--------------
Tiempo:  0.028569698333740234
Esperanza:  0.8354752019594158
Varianza:  0.019386808092052378
Desvio:  0.13923651852891317


Ejercicio 7:

In [ ]:
def Px(x):
  if x <= math.e and x >= 1:
    return 1/x
  else:
    return 0

def Py(y):
  return 1/(math.e - 1) #Densidad de la uniforme entre 1 y e.

def sim_y(): #Tiene que dar algo dentro del intervalo donde está definida.
  return random.uniform(1, math.e)

def transformadaInversaEj7(U):
  return (math.e)**(U)

def aceptacionRechazoEj7(sim_y,Px,Py,c):
  while True:
    y = sim_y()
    U = random.random()
    if U < (Px(y)/(Py(y)*c)):
      return y


constanteC = math.e - 1 #Máximo local que sirve de cota para el de aceptación y rechazo.
resultadoTrInv = lambda: transformadaInversaEj7(random.random())
resultadoAceptRechazo = lambda: aceptacionRechazoEj7(sim_y,Px,Py,constanteC)
start = time.time()

s, esperanza, _, varianza, desvio = simulate(resultadoTrInv,n_sim=10000)
prob_menor_que_dos = 1 - (np.sum(s > 2) / len(s))

print("--------------TEST Tr. Inv.--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)
print("Probabilidad de que X sea menor que 2: ", prob_menor_que_dos)

s, esperanza, _, varianza, desvio = simulate(resultadoAceptRechazo,n_sim=10000)
prob_menor_que_dos = 1 - (np.sum(s > 2) / len(s))

print("--------------TEST Acept. Rechazo--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)
print("Probabilidad de que X sea menor que 2: ", prob_menor_que_dos)

--------------TEST Tr. Inv.--------------
Tiempo:  0.004514455795288086
Esperanza:  1.7181215991351328
Varianza:  0.24597408317599534
Desvio:  0.4959577433370663
Probabilidad de que X sea menor que 2:  0.6893
--------------TEST Acept. Rechazo--------------
Tiempo:  0.018207788467407227
Esperanza:  1.7260966786883178
Varianza:  0.24390106532203235
Desvio:  0.49386340755519875
Probabilidad de que X sea menor que 2:  0.6841999999999999


Ejercicio 9)

a) generación de variables exponenciales según el ejemplo 5 f del libro Simulacion de S. M. Ross.

b) el método polar.

c) el método de razón entre uniformes.

In [ ]:
def composicion_new(ps: List[float], sims: List[Callable[[], float]]):

    assert len(ps) == len(sims) != 0
    assert sum(ps) == 1
    acum = []

    u = random()

    i, acc = 0, ps[0]

    while u >= acc:
        i += 1
        acc += ps[i]

    return sims[i]()

def normal_metodo_polar(mu, sigma):
  Rcuadrado = -2 * math.log(1 - random())
  Theta= 2 * math.pi * random()
  X= math.sqrt(Rcuadrado) * math.cos(Theta)
  Y= math.sqrt(Rcuadrado) * math.sin(Theta)
  return (X * sigma + mu, Y * sigma + mu)

def normal_ross(mu, sigma):
    # Primero genero |Z|

    def generarZ():

        while True:
            u = random()
            y = -math.log(random())

            if u <= math.exp(-((y-1)**2) / 2):
                return y

    generarMenosZ = lambda : -generarZ()

    # Después usamos composición para obtener Z a partir de componer |Z| y -|Z|
    return composicion_new([1/2, 1/2], [generarZ, generarMenosZ]) * sigma + mu

Ejercicio 10)

In [ ]:
func = (1/math.pi)**(1/2)

def cauchy(x):
  return (1/math.pi) * (1/((x**2) + 1)) #Es la funcion sin el lambda, que lo especifico después.

def ejercicio10():
  while True:
    U = random() * func
    V = func * (1-random()*2)
    if U**2 <= cauchy(V/U): #Por definición del método de razón.
      return V/U

Ejercicio 12:

Escriba un programa que calcule el número de eventos y sus tiempos de arribo en las primeras
T unidades de tiempo de un proceso de Poisson homogéneo con parámetro λ.


In [ ]:
def eventosPoisson(lamda,T):

 t=0
 Eventos=[]

 while t<T:
  t+=exponencial(lamda)
  if t<=T:
    Eventos.append(t)
 return len(Eventos), Eventos #Número de eventos y sus tiempos de llegada.

Ejercicio 13. Los autobuses que llevan los aficionados a un encuentro deportivo llegan a destino de acuerdo
con un proceso de Poisson a razón de cinco por hora. La capacidad de los autobuses es una variable
aleatoria que toma valores en el conjunto: {20,21,...,40} con igual probabilidad. A su vez, las capacidades
de dos autobuses distintos son variables independientes. Escriba un algoritmo para simular la llegada de
aficionados al encuentro en el instante t = 1hora.


In [ ]:
def capacidad():
  return random.randint(20,40)

def ejercicio13():

  cantidad_colectivos, _ = eventosPoisson(5,1)
  resultado = sum([capacidad() for _ in range(cantidad_colectivos)])
  return resultado

resultado2 = lambda: ejercicio13()
start = time.time()
s, esperanza, _, varianza, desvio = simulate(resultado2,n_sim=10000)

print("--------------TEST--------------")
print("Tiempo: ", time.time() - start)
print("Esperanza: ", esperanza)
print("Varianza: ", varianza)
print("Desvio: ", desvio)


--------------TEST--------------
Tiempo:  0.05759453773498535
Esperanza:  150.5516
Varianza:  4609.24273744
Desvio:  67.89140400256869


Ejercicio 14: Poisson con adelgazamiento mejorado que va variando la cota contra la que rechazo.

In [ ]:
def lam_i(t):
    # cota 7 (valor máximo cuando t = 0)
    if not 0 <= t <= 3:
        return 0
    return 3 + 4/(t+1)

def lam_ii(t):
    # cota 17
    if not 0 <= t <= 5:
        return 0
    return (t - 2)**2 - 5 * t + 17

def lam_iii(t):
    # cota 1/2
    if 2 <= t <= 3:
        return (t/2) - 1
    elif 3 <= t <= 6:
        return 1 - (t/6)
    else:
        return 0




Hot Dog


In [ ]:

def exponential(lamda):
    u = 1 - random.random()
    return -math.log(u)/lamda

def PoissonAdelgazamientoMejorado(T):

    def lamda_t(t):
        if 0 <= t < 3:
            return 5 + 5*t
        elif 3 <= t <= 5:
            return 20
        elif 5 < t <= 9:
            return 30-2*t

    interv = [1,2,6,8,9]
    lamdas = [10,15,18,14,12]
    intervals = [(10,1),(15,2),(18,6),(14,8),(18,9)]


    j = 0
    events = []
    t = exponential(lamda=intervals[j][0])
    while t <= T:
        lamda = intervals[j][0]
        b = intervals[j][1]
        if t <= b:
            v = random.random()
            if v < lamda_t(t)/lamda:
                events.append(t)
            t += exponential(lamda)

        else:  # t > b
            t = b + ((t-b) * lamda) / intervals[j+1][0]
            j += 1
    return len(events), events



n_sim = 10_000
arrivos = 0
T = 9
for _ in range(n_sim):
    arrivos += PoissonAdelgazamientoMejorado(T)[0]
print(f"esperanza: {arrivos/n_sim}")

esperanza: 132.074
